In [ ]:
# LOOP SIMPLE
# On recommence de zero sans crossover à chaque iter

ITER = 10
ATTACKED_CLASS = 1

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)

for i in range(ITER):
  pixel_to_change = pixel_to_change_generator(tensor_image)
  multiple_copies = multiple_copies_generator(tensor_image, 40)
  multiple_copies_changed = which_pixel_generator(multiple_copies, pixel_to_change)
  processed_copies = [transformIn(multiple_copies_changed[i]) for i in range(40)]
  input_copies = torch.stack(processed_copies) # -> les inputs qu'on va donné au CNN

  with torch.no_grad():
    outputs = model(input_copies)

  processed_copies = [transformOut(processed_copies[i]) for i in range(40)]
  logits = outputs.logits  # logits -> [40, num_classes]
  probabilities = torch.nn.functional.softmax(logits, dim=-1)
  predicted_classes = torch.argmax(probabilities, dim=-1)
  probabilities_attacked_class = probabilities[:, ATTACKED_CLASS]
  print(probabilities_attacked_class)


In [ ]:
# LOOP with CROSSOVER independant
# On recommence de zero à chaque iter

ITER = 10
ATTACKED_CLASS = 1

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)

for i in range(ITER):
  pixel_to_change = pixel_to_change_generator(tensor_image)
  multiple_copies = multiple_copies_generator(tensor_image, 40)
  multiple_copies_changed = which_pixel_generator(multiple_copies, pixel_to_change)
  parents_index = parent_generator(multiple_copies)
  multiple_copies_cross = crossover_generator(multiple_copies_changed, parents_index, pourcentage_height=0.15)

  processed_copies = [transformIn(multiple_copies_cross[i]) for i in range(40)]
  input_copies = torch.stack(processed_copies) # -> les inputs qu'on va donné au CNN

  with torch.no_grad():
    outputs = model(input_copies)

  processed_copies = [transformOut(processed_copies[i]) for i in range(40)]
  logits = outputs.logits  # logits -> [40, num_classes]
  probabilities = torch.nn.functional.softmax(logits, dim=-1)
  predicted_classes = torch.argmax(probabilities, dim=-1)
  probabilities_attacked_class = probabilities[:, ATTACKED_CLASS]
  print(probabilities_attacked_class)


In [ ]:
# LOOP with CROSSOVER dependant
# On modifie sur le crossover precedant à chaque iter

ITER = 10
ATTACKED_CLASS = 1

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)
multiple_copies = multiple_copies_generator(tensor_image, 40)

for i in range(ITER):
  # A chauqe iteration on modifie -> le nombre de pixels changé; le nombre de parents et quels parents sont utilisés
  pixel_to_change = pixel_to_change_generator(tensor_image)
  multiple_copies_changed = which_pixel_generator(multiple_copies, pixel_to_change)
  parents_index = parent_generator(multiple_copies)
  multiple_copies = crossover_generator(multiple_copies_changed, parents_index, pourcentage_height=0.15)

  processed_copies = [transformIn(multiple_copies[i]) for i in range(40)]
  input_copies = torch.stack(processed_copies) # -> les inputs qu'on va donné au CNN

  with torch.no_grad():
    outputs = model(input_copies)

  processed_copies = [transformOut(processed_copies[i]) for i in range(40)]
  logits = outputs.logits  # logits -> [40, num_classes]
  probabilities = torch.nn.functional.softmax(logits, dim=-1)
  predicted_classes = torch.argmax(probabilities, dim=-1)
  probabilities_attacked_class = probabilities[:, ATTACKED_CLASS]
  print(probabilities_attacked_class)


In [ ]:
# LOOP with CROSSOVER stagnant
# On fait un crossover du crossover sans modification à chaque iter

ITER = 10
ATTACKED_CLASS = 1

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)
multiple_copies = multiple_copies_generator(tensor_image, 40)
pixel_to_change = pixel_to_change_generator(tensor_image)
multiple_copies = which_pixel_generator(multiple_copies, pixel_to_change)

for i in range(ITER):
  parents_index = parent_generator(multiple_copies)
  multiple_copies = crossover_generator(multiple_copies, parents_index, pourcentage_height=0.15)

  processed_copies = [transformIn(multiple_copies[i]) for i in range(40)]
  input_copies = torch.stack(processed_copies) # -> les inputs qu'on va donné au CNN

  with torch.no_grad():
    outputs = model(input_copies)

  processed_copies = [transformOut(processed_copies[i]) for i in range(40)]
  logits = outputs.logits  # logits -> [40, num_classes]
  probabilities = torch.nn.functional.softmax(logits, dim=-1)
  predicted_classes = torch.argmax(probabilities, dim=-1)
  probabilities_attacked_class = probabilities[:, ATTACKED_CLASS]
  print(probabilities_attacked_class)


In [ ]:
# LOOP with CROSSOVER
# On fait un crossover du crossover sans modification à chaque iter
# On fait le crossover sur les 20 meilleurs index uniquement
# On garde 10 index aleatoires, 20 meilleur et 10 du crossover

ITER = 10

WANTED_CLASS = 0
UNWANTED_CLASS = 1

ELITES = 10

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)
multiple_copies = multiple_copies_generator(tensor_image, 40)
pixel_to_change = pixel_to_change_generator(tensor_image)
multiple_copies = which_pixel_generator(multiple_copies, pixel_to_change)

for i in range(ITER):
  processed_copies = [transformIn(multiple_copies[i]) for i in range(40)]
  input_copies = torch.stack(processed_copies) # -> les inputs qu'on va donné au CNN

  with torch.no_grad():
    outputs = model(input_copies)

  processed_copies = [transformOut(processed_copies[i]) for i in range(40)]

  logits = outputs.logits  # logits -> [40, num_classes]
  probabilities = torch.nn.functional.softmax(logits, dim=-1)

  predicted_classes = torch.argmax(probabilities, dim=-1) # to check if it worked

  # If we want the dog to be recognized as a cat -> controled attack
  probabilities_wanted_class = probabilities[:, WANTED_CLASS]
  sorted_probabilities, indices = torch.sort(probabilities_attacked_class, descending=True)

  # If we want the dog not to be recognized as a dog -> uncontroled attack
  # probabilities_unwanted_class = probabilities[:, UNWANTED_CLASS]
  # sorted_probabilities, indices = torch.sort(probabilities_attacked_class, descending=False)


  elites_index = sorted_probabilities[:ELITES]
  middle_index = sorted_probabilities[:(multiple_copies.shape[0]/2)]
  low_index = sorted_probabilities[(multiple_copies.shape[0]/2):]

  # 20 index
  top_selection = torch.cat((elites_index, middle_index))
  # 10 index
  random_low_index = low_index[torch.randperm(len(low_index))[:10]] # permutation alea. des index puis on selectionne simplement
  # 10 index
  parents_index = parent_generator(top_selection)
  crossed_copies = crossover_generator(top_selection, parents_index, pourcentage_height=0.15)

  # 40 index
  multiple_copies = torch.cat((top_selection, random_low_index, crossed_copies))

  print(predicted_classes)

# Only problem -> each iteration prediction are printed for the previous crossover

In [ ]:
# LOOP with CROSSOVER
# On fait un crossover du crossover sans modification à chaque iter
# On fait le crossover sur les 20 meilleurs index uniquement
# On garde 10 index aleatoires, 20 meilleur et 10 du crossover

ITER = 10

WANTED_CLASS = 0
UNWANTED_CLASS = 1

HEIGHT = 0.15
ELITES = 10

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)
multiple_copies = multiple_copies_generator(tensor_image, 40)
pixel_to_change = pixel_to_change_generator(tensor_image)
multiple_copies = which_pixel_generator(multiple_copies, pixel_to_change)

for i in range(ITER):
  probabilities = through_model(multiple_copies)
  elites_index, middle_index, low_index = selection(probabilities, wanted=True)

  predicted_classes = torch.argmax(probabilities, dim=-1) # to check if it worked

  # 20 index
  top_selection = torch.cat((elites_index, middle_index))
  # 10 index
  random_low_index = low_index[torch.randperm(len(low_index))[:10]] # permutation alea. des index puis on selectionne simplement
  # 10 index
  parents_index = parent_generator(top_selection)
  crossed_copies = crossover_generator(top_selection, parents_index, pourcentage_height=HEIGHT)

  # 40 index
  multiple_copies = torch.cat((top_selection, random_low_index, crossed_copies))

  print(predicted_classes)

# Only problem -> each iteration prediction are printed for the previous crossover

In [ ]:
# Check the base set before the crossover with YCbCr

# LOOP with CROSSOVER
# On fait un crossover du crossover sans modification à chaque iter
# On fait le crossover sur les 20 meilleurs index uniquement
# On garde 10 index aleatoires, 20 meilleur et 10 du crossover

ITER = 10

WANTED_CLASS = 0
UNWANTED_CLASS = 1

HEIGHT = 0.15
ELITES = 10

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)
tensor_image = from_rgb_to_ycbcr(tensor_image) # RGB -> YCbCr
multiple_copies = multiple_copies_generator(tensor_image, 40)
pixel_to_change = pixel_to_change_generator(tensor_image)
multiple_copies = which_pixel_generator(multiple_copies, pixel_to_change)

probabilities = through_model(multiple_copies)
elites_index, middle_index, low_index = selection(probabilities, wanted=True)
predicted_classes = torch.argmax(probabilities, dim=-1) # to check if it worked
print(predicted_classes) # check before the crossover

for i in range(ITER):

  # 20 index
  top_selection = torch.cat((elites_index, middle_index))
  # 10 index
  random_low_index = low_index[torch.randperm(len(low_index))[:10]] # permutation alea. des index puis on selectionne simplement
  # 10 index
  parents_index = parent_generator(top_selection)
  crossed_copies = crossover_generator(top_selection, parents_index, pourcentage_height=HEIGHT)

  # 40 index
  multiple_copies = torch.cat((top_selection, random_low_index, crossed_copies))

  probabilities = through_model(multiple_copies)
  elites_index, middle_index, low_index = selection(probabilities, wanted=True)

  predicted_classes = torch.argmax(probabilities, dim=-1) # to check if it worked

  print(predicted_classes)

In [ ]:
# ⚠️ RUN FROM HERE ⚠️

In [ ]:
from transformers import AutoModelForImageClassification, AutoProcessor
import torch
import torchvision.transforms as transforms
from PIL import Image
import requests
import matplotlib.pyplot as plt
import random
import numpy as np
import sys


In [ ]:
image = Image.open("/content/sample_data/dog.bmp")

plt.imshow(image)
plt.axis("off")
plt.show

In [ ]:
from transformers import AutoImageProcessor

model_name = "hilmansw/resnet18-catdog-classifier"

model = AutoModelForImageClassification.from_pretrained(model_name)
model.eval()

# processor = AutoProcessor.from_pretrained(model_name)
# print(processor)

In [ ]:
# Les Transforms

transformTensor = transforms.ToTensor()

transformPIL = transforms.ToPILImage()

transformIn = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),  # -> je ne suis pas sûr de moi
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalisation -> ImageNet
])

transformOut = transforms.Compose([
    transforms.Normalize(
        mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
        std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
    )
])

In [ ]:
# Mr. TOPAL approach

def pixel_to_change_generator(image):
    channels, height, width = image.shape
    n = 60 # Defini la range de valeur possible -> generalement proche de 1
    factor = torch.rand(1).item() # Defini ou on se trouve dans cette range
    res = (factor ** (1.0 / (n + 1))) * (height * width)
    pixel_to_change = (height * width) - res
    return pixel_to_change

In [ ]:
# My approach

def pixel_to_change_generator(image, lower_pourcentage = 0.001, upper_pourcentage = 0.01): # -> Change between 0.1% and max 1% of the image, the goal is to lower upper_pourcentage
    channels, height, width = image.shape
    random_pourcentage = random.uniform(lower_pourcentage, upper_pourcentage)
    pixel_to_change =  random_pourcentage * (height * width)
    pixel_to_change = int(pixel_to_change)

    return pixel_to_change, random_pourcentage

In [ ]:
# My approach without randomness

def pixel_to_change_calculator(image, pourcentage = 0.01): # -> Change between 0.1% and max 1% of the image, the goal is to lower upper_pourcentage
    channels, height, width = image.shape
    pixel_to_change =  pourcentage * (height * width)
    pixel_to_change = int(pixel_to_change)

    return pixel_to_change

In [ ]:
def multiple_copies_generator(image, number_of_copies = 40):
    multiple_copies = torch.stack([image] * number_of_copies) # -> Format [number_of_copies, 3, x, y]
    return multiple_copies

In [ ]:
# Genere aleatoirement les valeurs a ajouter sur chaque pixel

def which_pixel_generator(multiple_copies, pixel_to_change, max_value = 1): # reach -> 0 to 1

    multiple_copies_changed = multiple_copies.clone()

    batch_size, channels, height, width = multiple_copies.shape
    pixel_c = torch.randint(0, channels, (batch_size, pixel_to_change))
    pixel_x = torch.randint(0, height, (batch_size, pixel_to_change))
    pixel_y = torch.randint(0, width, (batch_size, pixel_to_change))

    # PIL -> Tensor = automaticaly normalised between 0 and 1
    values_to_change = (torch.rand(batch_size, pixel_to_change) * 2 - 1) * max_value

    multiple_copies_changed[torch.arange(batch_size).unsqueeze(1), pixel_c, pixel_x, pixel_y] += values_to_change

    return torch.clamp(multiple_copies_changed, 0, 1)

In [ ]:
# Donne directement la valeur a ajouter sur les pixels

def which_pixel_calculator(multiple_copies, pixel_to_change, value = 0.1):

    multiple_copies_changed = multiple_copies.clone()

    batch_size, channels, height, width = multiple_copies.shape
    pixel_c = torch.randint(0, channels, (batch_size, pixel_to_change))
    pixel_x = torch.randint(0, height, (batch_size, pixel_to_change))
    pixel_y = torch.randint(0, width, (batch_size, pixel_to_change))

    # PIL -> Tensor = automaticaly normalised between 0 and 1
    values_to_change = (torch.randint(0, 2, (batch_size, pixel_to_change)) * 2 - 1) * value

    multiple_copies_changed[torch.arange(batch_size).unsqueeze(1), pixel_c, pixel_x, pixel_y] += values_to_change

    return torch.clamp(multiple_copies_changed, 0, 1)

In [ ]:
# à modifier plus tard avec  les changements du haut

def which_pixel_generator_targeted(multiple_copies, pixel_to_change, targeted_channel): # 2 for blue channel

    multiple_copies_changed = multiple_copies.clone()
    pixel_c = torch.full((multiple_copies.shape[0], pixel_to_change), targeted_channel)
    pixel_x = torch.randint(0, multiple_copies.shape[2], (multiple_copies.shape[0], pixel_to_change))
    pixel_y = torch.randint(0, multiple_copies.shape[3], (multiple_copies.shape[0], pixel_to_change))
    values_to_change = torch.rand(multiple_copies.shape[0], pixel_to_change)

    multiple_copies_changed[torch.arange(multiple_copies.shape[0]).unsqueeze(1), pixel_c, pixel_x, pixel_y] = values_to_change # -> unsqueeze is necessary to have a proper broadcast
    return multiple_copies_changed

In [ ]:
def parent_generator(multiple_copies):
        batch_size, channels, height, width = multiple_copies.shape
        number_of_parents = random.randrange(0, batch_size, 2)
        parents_index = random.sample(range(0, batch_size), number_of_parents)
        return parents_index


In [ ]:
def crossover_generator(multiple_copies, parents_index, pourcentage_height = 0.15):
  multiple_copies_cross = multiple_copies.clone()

  batch_size, channels, height, width = multiple_copies.shape

  for i in range(0, len(parents_index), 2):
    parent_index_1 = parents_index[i]
    parent_index_2 = parents_index[i + 1]

    max_size_square = int(multiple_copies.shape[3] * pourcentage_height)  # % hauteur de l'image -> multiple_copies[0].shape[2] prend le nombre de pixel d'une image

    # Definition de la zone

    which_channel = random.randint(0, channels - 1)  # -1 pour éviter l'out of bounds
    size_height = random.randint(0, max_size_square)
    start_height = random.randint(0, height - size_height)
    size_width = random.randint(0, max_size_square)
    start_width = random.randint(0, width - size_width)

    # Echange
    zone_save = multiple_copies_cross[parent_index_1, which_channel, start_height:start_height + size_height, start_width:start_width + size_width,].clone()

    # Échange des zones entre les deux images
    multiple_copies_cross[parent_index_1, which_channel, start_height:start_height + size_height, start_width:start_width + size_width] = \
        multiple_copies_cross[parent_index_2, which_channel, start_height:start_height + size_height, start_width:start_width + size_width]

    multiple_copies_cross[parent_index_2, which_channel, start_height:start_height + size_height, start_width:start_width + size_width] = zone_save

  return multiple_copies_cross

In [ ]:
def through_model(multiple_copies):

  processed_copies = [transformIn(multiple_copies[i]) for i in range(40)]
  input_copies = torch.stack(processed_copies) # -> les inputs qu'on va donné au CNN

  with torch.no_grad():
    outputs = model(input_copies)

  processed_copies = [transformOut(processed_copies[i]) for i in range(40)]

  logits = outputs.logits  # logits -> [40, num_classes]
  probabilities = torch.nn.functional.softmax(logits, dim=-1)

  return probabilities

In [ ]:
def selection(multiple_copies, probabilities, elites, wanted_class, unwanted_class, wanted: True):

  batch_size, channels, height, width = multiple_copies.shape

  if wanted:
    # If we want the dog to be recognized as a cat -> controled attack
    probabilities_wanted_class = probabilities[:, wanted_class]
    sorted_probabilities, indices = torch.sort(probabilities_wanted_class, descending=True)

  else:
    #If we want the dog not to be recognized as a dog -> uncontroled attack
    probabilities_unwanted_class = probabilities[:, wanted_class]
    sorted_probabilities, indices = torch.sort(probabilities_unwanted_class, descending=False)

  middle = batch_size//2

  elites_index = sorted_probabilities[:elites]
  middle_index = sorted_probabilities[:middle]
  low_index = sorted_probabilities[middle:]

  # Convertir pour Pytorch
  elites_index = torch.tensor(elites_index, dtype=torch.long)
  middle_index = torch.tensor(middle_index, dtype=torch.long)
  low_index = torch.tensor(low_index, dtype=torch.long)

  elites_selection = multiple_copies[elites_index]
  middle_selection = multiple_copies[middle_index]
  low_selection = multiple_copies[low_index]

  return elites_selection, middle_selection, low_selection

In [ ]:
def from_rgb_to_ycbcr(tensor_image):

  r = tensor_image[0, :, :]
  g = tensor_image[1, :, :]
  b = tensor_image[2, :, :]

  y = 0.299 * r + 0.587 * g + 0.114 * b
  cb = -0.169 * r - 0.331 * g + 0.499 * b
  cr = 0.499 * r - 0.460 * g - 0.039 * b

  tensor_image_ycbcr = torch.stack((y, cb, cr), dim=0)

  return tensor_image_ycbcr

In [ ]:
def checker(probabilities):
  predicted_classes = torch.argmax(probabilities, dim=-1) # to check if it worked
  wanted_index = (predicted_classes == WANTED_CLASS).nonzero(as_tuple=True)[0] # first to boolean tensor -> only keep True index -> keep dimension 0!
  if wanted_index.nelement() > 0:
    return True, wanted_index
  else:
    return False, wanted_index

In [ ]:
# Check the base set before the crossover

# LOOP with CROSSOVER
# On fait un crossover du crossover sans modification à chaque iter
# On fait le crossover sur les 20 meilleurs index uniquement
# On garde 10 index aleatoires, 20 meilleur et 10 du crossover

ITER = 10

WANTED_CLASS = 0
UNWANTED_CLASS = 1

HEIGHT = 0.15
ELITES = 10

RANGE = [0.3, 0.4, 0.45]

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)
multiple_copies = multiple_copies_generator(tensor_image, 40)

adv_found = False

for actual_range in RANGE:

  pixel_to_change = pixel_to_change_generator(tensor_image, upper_pourcentage=actual_range)
  multiple_copies = which_pixel_generator(multiple_copies, pixel_to_change)

  probabilities = through_model(multiple_copies)
  elites_selection, middle_selection, low_selection = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)
  predicted_classes = torch.argmax(probabilities, dim=-1) # to check if it worked
  print(predicted_classes) # check before the crossover

  # DEBUG

  # DEBUG

  for i in range(ITER):

    # 20 index
    top_selection = torch.cat((elites_selection, middle_selection))
    # 10 index
    random_low_selection = low_selection[torch.randperm(len(low_selection))[:10]] # permutation alea. des index puis on selectionne simplement
    # 10 index
    parents_index = parent_generator(top_selection)
    crossed_copies = crossover_generator(top_selection, parents_index, pourcentage_height=HEIGHT)

    # 40 index
    multiple_copies = torch.cat((top_selection, random_low_selection, crossed_copies))

    probabilities = through_model(multiple_copies)
    elites_selection, middle_selection, low_selection = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)

    success, wanted_index = checker(probabilities)
    if success:
      adv_found = True
      break

  if adv_found:
    break

adv_images = multiple_copies[list(wanted_index)]

for adv_image in adv_images:
  image_restored = transformPIL(adv_image)
  plt.imshow(image_restored)
  plt.axis("off")
  plt.show

In [ ]:
# WHILE LOOP

# Check the base set before the crossover

# LOOP with CROSSOVER
# On fait un crossover du crossover sans modification à chaque iter
# On fait le crossover sur les 20 meilleurs index uniquement
# On garde 10 index aleatoires, 20 meilleur et 10 du crossover

ITER = 10

WANTED_CLASS = 0
UNWANTED_CLASS = 1

HEIGHT = 0.15
ELITES = 10

ACTUAL_POURCENTAGE = 0

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)
multiple_copies = multiple_copies_generator(tensor_image, 40)

adv_found = False

while not adv_found and ACTUAL_POURCENTAGE < 1:

  ACTUAL_POURCENTAGE += 0.05

  # pixel_to_change = pixel_to_change_generator(tensor_image, upper_pourcentage=ACTUAL_POURCENTAGE)
  pixel_to_change = pixel_to_change_calculator(tensor_image, pourcentage=ACTUAL_POURCENTAGE) # modify a fix number of pixel, no randomness

  multiple_copies = which_pixel_generator(multiple_copies, pixel_to_change)

  probabilities = through_model(multiple_copies)
  elites_selection, middle_selection, low_selection = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)
  predicted_classes = torch.argmax(probabilities, dim=-1) # to check if it worked
  print(predicted_classes) # check before the crossover

  for i in range(ITER):

    # 20 index
    top_selection = torch.cat((elites_selection, middle_selection))
    # 10 index
    random_low_selection = low_selection[torch.randperm(len(low_selection))[:10]] # permutation alea. des index puis on selectionne simplement
    # 10 index
    parents_index = parent_generator(top_selection)
    crossed_copies = crossover_generator(top_selection, parents_index, pourcentage_height=HEIGHT)

    # 40 index
    multiple_copies = torch.cat((top_selection, random_low_selection, crossed_copies))

    probabilities = through_model(multiple_copies)
    elites_selection, middle_selection, low_selection = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)

    success, wanted_index = checker(probabilities)
    if success:
      adv_found = True
      break


if(ACTUAL_POURCENTAGE >= 1):
  print("NOTHING WAS FOUND")
elif(adv_found == True):
  print("ADV IMAGE FOUND!")
  print(f"ACTUAL_POURCENTAGE: {ACTUAL_POURCENTAGE}")

adv_images = multiple_copies[list(wanted_index)]

for adv_image in adv_images:
  image_restored = transformPIL(adv_image)
  plt.imshow(image_restored)
  plt.axis("off")
  plt.show

In [ ]:
# NUMBER OF PIXELS -> OPTI
# PIXELS MODIFICATION -> FIXED

ITER = 10

WANTED_CLASS = 0
UNWANTED_CLASS = 1

HEIGHT = 0.15
ELITES = 10

ACTUAL_POURCENTAGE = 0

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)
multiple_copies = multiple_copies_generator(tensor_image, 40)

adv_found = False

while not adv_found and ACTUAL_POURCENTAGE < 1:

  ACTUAL_POURCENTAGE += 0.05

  pixel_to_change = pixel_to_change_calculator(tensor_image, pourcentage=ACTUAL_POURCENTAGE) # modify a fix number of pixel, no randomness

  multiple_copies = which_pixel_calculator(multiple_copies, pixel_to_change, 0.5)

  probabilities = through_model(multiple_copies)
  elites_selection, middle_selection, low_selection = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)
  predicted_classes = torch.argmax(probabilities, dim=-1) # to check if it worked
  print(predicted_classes) # check before the crossover

  for i in range(ITER):

    # 20 index
    top_selection = torch.cat((elites_selection, middle_selection))
    # 10 index
    random_low_selection = low_selection[torch.randperm(len(low_selection))[:10]] # permutation alea. des index puis on selectionne simplement
    # 10 index
    parents_index = parent_generator(top_selection)
    crossed_copies = crossover_generator(top_selection, parents_index, pourcentage_height=HEIGHT)

    # 40 index
    multiple_copies = torch.cat((top_selection, random_low_selection, crossed_copies))

    probabilities = through_model(multiple_copies)
    elites_selection, middle_selection, low_selection = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)

    success, wanted_index = checker(probabilities)
    if success:
      adv_found = True
      break


if(ACTUAL_POURCENTAGE >= 1):
  print("NOTHING WAS FOUND")
elif(adv_found == True):
  print("ADV IMAGE FOUND!")
  print(f"ACTUAL_POURCENTAGE: {ACTUAL_POURCENTAGE}")

adv_images = multiple_copies[list(wanted_index)]

for adv_image in adv_images:
  image_restored = transformPIL(adv_image)
  plt.imshow(image_restored)
  plt.axis("off")
  plt.show

In [ ]:
# NUMBER OF PIXELS -> FIXED
# PIXELS MODIFICATION -> OPTI

ITER = 10

WANTED_CLASS = 0
UNWANTED_CLASS = 1

HEIGHT = 0.15
ELITES = 10

ACTUAL_MODIFICATION = 1

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)
multiple_copies = multiple_copies_generator(tensor_image, 40)

adv_found = False

while not adv_found and ACTUAL_MODIFICATION > 0:

  ACTUAL_MODIFICATION -= 0.05

  pixel_to_change = pixel_to_change_calculator(tensor_image, pourcentage=0.3) # modify a fix number of pixel, no randomness

  multiple_copies = which_pixel_calculator(multiple_copies, pixel_to_change, ACTUAL_MODIFICATION)

  probabilities = through_model(multiple_copies)
  elites_selection, middle_selection, low_selection = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)
  predicted_classes = torch.argmax(probabilities, dim=-1) # to check if it worked
  print(predicted_classes) # check before the crossover

  for i in range(ITER):

    # 20 index
    top_selection = torch.cat((elites_selection, middle_selection))
    # 10 index
    random_low_selection = low_selection[torch.randperm(len(low_selection))[:10]] # permutation alea. des index puis on selectionne simplement
    # 10 index
    parents_index = parent_generator(top_selection)
    crossed_copies = crossover_generator(top_selection, parents_index, pourcentage_height=HEIGHT)

    # 40 index
    multiple_copies = torch.cat((top_selection, random_low_selection, crossed_copies))

    probabilities = through_model(multiple_copies)
    elites_selection, middle_selection, low_selection = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)

    success, wanted_index = checker(probabilities)
    if success:
      adv_found = True
      break


if(ACTUAL_MODIFICATION <= 0):
  print("NOTHING WAS FOUND")
elif(adv_found == True):
  print("ADV IMAGE FOUND!")
  print(f"ACTUAL_MODIFICATION: {ACTUAL_MODIFICATION}")

adv_images = multiple_copies[list(wanted_index)]

for adv_image in adv_images:
  image_restored = transformPIL(adv_image)
  plt.imshow(image_restored)
  plt.axis("off")
  plt.show

In [ ]:
# BEST VALUES FOUND
# INTRODUCING RANDOMNESS

# NUMBER OF PIXELS -> MAX
# PIXELS MODIFICATION -> MAX

ITER = 20

WANTED_CLASS = 0
UNWANTED_CLASS = 1

HEIGHT = 0.15
ELITES = 10

best_diff = sys.maxsize

image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image)
multiple_copies = multiple_copies_generator(tensor_image, 40)

pixel_to_change = pixel_to_change_generator(tensor_image, upper_pourcentage=ACTUAL_POURCENTAGE) # modify a fix number of pixel, no randomness

multiple_copies = which_pixel_generator(multiple_copies, pixel_to_change, ACTUAL_MODIFICATION)

probabilities = through_model(multiple_copies)
elites_selection, middle_selection, low_selection = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)
predicted_classes = torch.argmax(probabilities, dim=-1) # to check if it worked
print(predicted_classes) # check before the crossover

for i in range(ITER):

  # 20 index
  top_selection = torch.cat((elites_selection, middle_selection))
  # 10 index
  random_low_selection = low_selection[torch.randperm(len(low_selection))[:10]] # permutation alea. des index puis on selectionne simplement
  # 10 index
  parents_index = parent_generator(top_selection)
  crossed_copies = crossover_generator(top_selection, parents_index, pourcentage_height=HEIGHT)

  # 40 index
  multiple_copies = torch.cat((top_selection, random_low_selection, crossed_copies))

  probabilities = through_model(multiple_copies)
  elites_selection, middle_selection, low_selection = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)

  success, wanted_index = checker(probabilities)
  if success:
    adv_images = multiple_copies[list(wanted_index)]
    for adv_image in adv_images:
      diff = tensor_image - adv_image
      if torch.sum(diff).item() < best_diff:
        best_image = adv_image

# Display best image

image_restored = transformPIL(best_image)
plt.imshow(image_restored)
plt.axis("off")
plt.show

In [ ]:
# Next with YCbCr individual channels targeted